In [ ]:
# This cell mounts your Google Drive in Colab so we can access the saved Gemma adapter and evaluation files.

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# This cell logs into Hugging Face if needed and exposes the saved token to subprocesses so the evaluator can download the gated Gemma base model.

import os
from huggingface_hub import notebook_login, whoami, get_token

try:
    account = whoami()
except Exception:
    notebook_login()
    account = whoami()

hf_token = get_token()

if not hf_token:
    raise RuntimeError("No Hugging Face token was found after login.")

os.environ["HF_TOKEN"] = hf_token

print("Logged in as:", account["name"])
print("HF token available to evaluator:", bool(os.environ.get("HF_TOKEN")))
print("✅ Hugging Face authentication is ready for subprocess evaluation.")

Logged in as: Poojitha1997
HF token available to evaluator: True
✅ Hugging Face authentication is ready for subprocess evaluation.


In [ ]:
# This cell checks how many AFTER-SFT GSM8K predictions have already been saved to Google Drive so we know exactly where the evaluation can resume.

import os
import glob
import json

RESULTS_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "gemma_after_sft_fp16_full"
)

prediction_files = glob.glob(
    os.path.join(RESULTS_DIR, "*fp16_full_final*_predictions.jsonl")
)

print("=" * 80)
print("AFTER-SFT SAVED PROGRESS CHECK")
print("=" * 80)

if not prediction_files:
    print("❌ No prediction file found.")
    print("Checked folder:")
    print(RESULTS_DIR)
else:
    for prediction_file in prediction_files:
        valid_records = 0
        completed_without_error = 0
        generation_errors = 0
        unique_ids = set()

        with open(prediction_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)
                    valid_records += 1

                    problem_id = record.get("problem_id")
                    if problem_id:
                        unique_ids.add(problem_id)

                    if record.get("generation_error") is None:
                        completed_without_error += 1
                    else:
                        generation_errors += 1

                except json.JSONDecodeError:
                    pass

        total = 1319
        completed = len(unique_ids)
        remaining = total - completed
        percent = (completed / total) * 100

        print("\nPrediction file:")
        print(prediction_file)

        print("\nSaved records:", valid_records)
        print("Unique GSM8K examples saved:", completed)
        print("Completed without generation error:", completed_without_error)
        print("Generation-error records:", generation_errors)

        print("\nProgress:")
        print(f"{completed} / {total}")
        print(f"{percent:.2f}% completed")
        print(f"{remaining} examples remaining")

        if completed > 0:
            print("\n✅ Your completed evaluation work is saved.")
            print("You can resume using the SAME full-evaluation cell.")
        else:
            print("\n⚠️ Prediction file exists, but no completed examples were found.")

AFTER-SFT SAVED PROGRESS CHECK

Prediction file:
/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/gemma_after_sft_fp16_full/google_gemma-3-1b-it_after_sft_3a9e3d53_fp16_full_final_predictions.jsonl

Saved records: 1146
Unique GSM8K examples saved: 1136
Completed without generation error: 1146
Generation-error records: 0

Progress:
1136 / 1319
86.13% completed
183 examples remaining

✅ Your completed evaluation work is saved.
You can resume using the SAME full-evaluation cell.


In [ ]:
# This cell upgrades TorchAO to version 0.17.0 so PEFT can correctly load the trained Gemma LoRA adapter with PyTorch 2.11.

%pip install -q --upgrade "torchao==0.17.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.6 MB/s eta 0:00:00


In [ ]:
# This cell runs a 5-question GSM8K smoke evaluation using FP16 Gemma 3 1B plus the best trained LoRA adapter to verify that after-SFT inference works correctly.

import os
import subprocess
import sys

EVALUATOR_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Distillation_Evaluation_Code/evaluate_gsm8k_latest.py"
)

ADAPTER_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "gemma3_1b_qlora_v1/final_best_adapter"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "gemma_after_sft_fp16_smoke"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

command = [
    sys.executable,
    EVALUATOR_PATH,

    "--backend", "transformers",
    "--model", "google/gemma-3-1b-it",
    "--adapter-path", ADAPTER_PATH,
    "--stage", "after_sft",

    "--limit", "5",

    "--max-input-tokens", "1536",
    "--max-new-tokens", "768",

    "--dtype", "float16",

    "--output-dir", RESULTS_DIR,
    "--run-tag", "fp16_smoke",
]

print("=" * 80)
print("GEMMA AFTER-SFT SMOKE EVALUATION")
print("=" * 80)
print("Model: google/gemma-3-1b-it")
print("Adapter:", ADAPTER_PATH)
print("Questions: 5")
print("Precision: FP16")
print("Max input tokens: 1536")
print("Max new tokens: 768")
print("4-bit evaluation: OFF")
print("=" * 80)

subprocess.run(command, check=True)

GEMMA AFTER-SFT SMOKE EVALUATION
Model: google/gemma-3-1b-it
Adapter: /content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/gemma3_1b_qlora_v1/final_best_adapter
Questions: 5
Precision: FP16
Max input tokens: 1536
Max new tokens: 768
4-bit evaluation: OFF


KeyboardInterrupt: 

In [ ]:
# This cell runs the official full AFTER-SFT GSM8K evaluation of the trained Gemma 3 1B model using the best LoRA adapter, FP16 inference, and records total wall-clock runtime.

import os
import sys
import time
import subprocess
from datetime import datetime

EVALUATOR_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Distillation_Evaluation_Code/evaluate_gsm8k_latest.py"
)

ADAPTER_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "gemma3_1b_qlora_v1/final_best_adapter"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "gemma_after_sft_fp16_full"
)

if not os.path.isfile(EVALUATOR_PATH):
    raise FileNotFoundError(f"Evaluator not found: {EVALUATOR_PATH}")

if not os.path.isdir(ADAPTER_PATH):
    raise FileNotFoundError(f"Adapter not found: {ADAPTER_PATH}")

if not os.path.isfile(os.path.join(ADAPTER_PATH, "adapter_model.safetensors")):
    raise FileNotFoundError("adapter_model.safetensors not found.")

if not os.path.isfile(os.path.join(ADAPTER_PATH, "adapter_config.json")):
    raise FileNotFoundError("adapter_config.json not found.")

os.makedirs(RESULTS_DIR, exist_ok=True)

command = [
    sys.executable,
    EVALUATOR_PATH,
    "--backend", "transformers",
    "--model", "google/gemma-3-1b-it",
    "--adapter-path", ADAPTER_PATH,
    "--stage", "after_sft",
    "--max-input-tokens", "1536",
    "--max-new-tokens", "768",
    "--dtype", "float16",
    "--output-dir", RESULTS_DIR,
    "--run-tag", "fp16_full_final",
]

print("=" * 80)
print("GEMMA 3 1B — OFFICIAL AFTER-SFT FULL GSM8K EVALUATION")
print("=" * 80)
print("Evaluator:")
print(EVALUATOR_PATH)
print()
print("Base model:        google/gemma-3-1b-it")
print("LoRA adapter:      final_best_adapter")
print("Stage:             after_sft")
print("Dataset:           GSM8K official test")
print("Expected examples: 1,319")
print("Precision:         FP16")
print("Max input tokens:  1536")
print("Max new tokens:    768")
print("4-bit evaluation:  OFF")
print("Limit:             NONE — full test set")
print()
print("Results directory:")
print(RESULTS_DIR)
print("=" * 80)

start_time = datetime.now()
timer_start = time.perf_counter()

print("\nStart time:", start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("\nStarting evaluation...\n")

try:
    subprocess.run(
        command,
        check=True,
        env=os.environ.copy()
    )

    success = True

except subprocess.CalledProcessError as exc:
    success = False
    print("\n❌ Evaluation process failed.")
    print("Return code:", exc.returncode)
    raise

finally:
    timer_end = time.perf_counter()
    end_time = datetime.now()

    elapsed_seconds = timer_end - timer_start
    elapsed_minutes = elapsed_seconds / 60
    elapsed_hours = elapsed_seconds / 3600

    print("\n" + "=" * 80)
    print("FULL EVALUATION RUNTIME")
    print("=" * 80)
    print("Start time:      ", start_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("End time:        ", end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print(f"Elapsed seconds:  {elapsed_seconds:.2f}")
    print(f"Elapsed minutes:  {elapsed_minutes:.2f}")
    print(f"Elapsed hours:    {elapsed_hours:.2f}")
    print("=" * 80)

    if success:
        print("\n✅ Official AFTER-SFT evaluation completed successfully.")
        print("Results saved in:")
        print(RESULTS_DIR)

GEMMA 3 1B — OFFICIAL AFTER-SFT FULL GSM8K EVALUATION
Evaluator:
/content/drive/MyDrive/Colab Notebooks/Distillation_Evaluation_Code/evaluate_gsm8k_latest.py

Base model:        google/gemma-3-1b-it
LoRA adapter:      final_best_adapter
Stage:             after_sft
Dataset:           GSM8K official test
Expected examples: 1,319
Precision:         FP16
Max input tokens:  1536
Max new tokens:    768
4-bit evaluation:  OFF
Limit:             NONE — full test set

Results directory:
/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/gemma_after_sft_fp16_full

Start time: 2026-08-12 16:33:16

Starting evaluation...


FULL EVALUATION RUNTIME
Start time:       2026-08-12 16:33:16
End time:         2026-08-12 17:26:07
Elapsed seconds:  3170.29
Elapsed minutes:  52.84
Elapsed hours:    0.88

✅ Official AFTER-SFT evaluation completed successfully.
Results saved in:
/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/gemma_after_sft_fp16_full


In [ ]:
# This cell evaluates all saved Gemma checkpoints on the 485-example validation set using your exact evaluator logic and ranks them by validation exact-match accuracy.

import os
import gc
import sys
import json
import glob
import importlib.util
from pathlib import Path

import torch

EVALUATOR_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Distillation_Evaluation_Code/evaluate_gsm8k_latest.py"
)

VAL_FILE = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "data_final_v3/"
    "teacher_gsm8k_val_qwen3_14b_awq_gsm8k_teacher_v4_434a9551e7_full_sft.jsonl"
)

TRAINING_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "gemma3_1b_qlora_v1"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "gemma_checkpoint_validation_eval"
)

# -------------------------------------------------------------------------
# Verify required paths
# -------------------------------------------------------------------------

if not os.path.isfile(EVALUATOR_PATH):
    raise FileNotFoundError(f"Evaluator not found: {EVALUATOR_PATH}")

if not os.path.isfile(VAL_FILE):
    raise FileNotFoundError(f"Validation file not found: {VAL_FILE}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# Find saved checkpoints
# -------------------------------------------------------------------------

checkpoint_paths = sorted(
    glob.glob(os.path.join(TRAINING_DIR, "checkpoint-*")),
    key=lambda p: int(Path(p).name.split("-")[-1])
)

if not checkpoint_paths:
    raise RuntimeError("No checkpoint-* folders were found.")

print("=" * 80)
print("CHECKPOINTS FOUND")
print("=" * 80)

for checkpoint in checkpoint_paths:
    print(Path(checkpoint).name)

# -------------------------------------------------------------------------
# Safely import your evaluator module
# -------------------------------------------------------------------------

module_name = "gsm8k_evaluator"

spec = importlib.util.spec_from_file_location(
    module_name,
    EVALUATOR_PATH
)

evaluator = importlib.util.module_from_spec(spec)

# Required so dataclass definitions inside the evaluator load correctly
sys.modules[module_name] = evaluator

spec.loader.exec_module(evaluator)

print("\n✅ Evaluator imported successfully.")

# -------------------------------------------------------------------------
# Load the 485 validation examples
# -------------------------------------------------------------------------

examples = []

with open(VAL_FILE, "r", encoding="utf-8") as f:
    for index, line in enumerate(f):
        line = line.strip()

        if not line:
            continue

        row = json.loads(line)

        examples.append(
            {
                "problem_id": str(row["problem_id"]),
                "source_index": index,
                "question": str(row["question"]),
                "gold_answer": str(row["gold_answer"]),
            }
        )

print("Validation examples loaded:", len(examples))

if len(examples) != 485:
    raise ValueError(
        f"Expected 485 validation examples, but found {len(examples)}"
    )

# -------------------------------------------------------------------------
# Evaluate every saved checkpoint
# -------------------------------------------------------------------------

results = []

for checkpoint_path in checkpoint_paths:

    checkpoint_name = Path(checkpoint_path).name

    print("\n" + "=" * 80)
    print("EVALUATING:", checkpoint_name)
    print("=" * 80)

    config = evaluator.EvalConfig(
        backend="transformers",
        model="google/gemma-3-1b-it",
        stage="validation_checkpoint_selection",
        adapter_path=checkpoint_path,
        base_url=None,
        limit=None,
        max_new_tokens=768,
        max_input_tokens=1536,
        load_in_4bit=False,
        dtype="float16",
        trust_remote_code=False,
        disable_qwen_thinking=True,
    )

    run_slug = f"{checkpoint_name}_{config.config_hash()}"

    predictions_path = (
        OUTPUT_DIR / f"{run_slug}_predictions.jsonl"
    )

    metrics_path = (
        OUTPUT_DIR / f"{run_slug}_metrics.json"
    )

    backend = evaluator.create_backend(
        config,
        api_key="EMPTY",
        timeout_seconds=180.0,
    )

    metrics = evaluator.evaluate_dataset(
        config=config,
        backend=backend,
        examples=examples,
        predictions_path=predictions_path,
        metrics_path=metrics_path,
    )

    results.append(
        {
            "checkpoint": checkpoint_name,
            "checkpoint_path": checkpoint_path,
            "accuracy": metrics["exact_match_accuracy"],
            "valid_format_rate": metrics["valid_format_rate"],
            "correct_and_valid_rate": metrics["correct_and_valid_rate"],
            "truncation_rate": metrics["truncation_rate"],
            "generation_error_rate": metrics["generation_error_rate"],
            "metrics_file": str(metrics_path),
        }
    )

    print(
        f"\n{checkpoint_name} validation accuracy: "
        f"{metrics['exact_match_accuracy']:.2%}"
    )

    print(
        f"Valid format: "
        f"{metrics['valid_format_rate']:.2%}"
    )

    print(
        f"Correct + valid: "
        f"{metrics['correct_and_valid_rate']:.2%}"
    )

    # Free GPU memory before loading the next checkpoint
    del backend
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# Rank checkpoints by validation exact-match accuracy
# -------------------------------------------------------------------------

results.sort(
    key=lambda x: x["accuracy"],
    reverse=True
)

print("\n" + "=" * 80)
print("VALIDATION CHECKPOINT RANKING")
print("=" * 80)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. {result['checkpoint']} | "
        f"Accuracy: {result['accuracy']:.2%} | "
        f"Valid format: {result['valid_format_rate']:.2%} | "
        f"Correct+Valid: {result['correct_and_valid_rate']:.2%} | "
        f"Truncated: {result['truncation_rate']:.2%} | "
        f"Generation errors: {result['generation_error_rate']:.2%}"
    )

best = results[0]

print("\n" + "=" * 80)
print("BEST CHECKPOINT BY VALIDATION EXACT-MATCH")
print("=" * 80)

print("Checkpoint:", best["checkpoint"])
print("Path:", best["checkpoint_path"])
print(f"Validation accuracy: {best['accuracy']:.2%}")
print(f"Valid format: {best['valid_format_rate']:.2%}")
print(f"Correct + valid: {best['correct_and_valid_rate']:.2%}")

print("\nMetrics file:")
print(best["metrics_file"])

print("=" * 80)

CHECKPOINTS FOUND
checkpoint-242
checkpoint-363

✅ Evaluator imported successfully.
Validation examples loaded: 485

EVALUATING: checkpoint-242


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Evaluating google/gemma-3-1b-it [validation_checkpoint_selection] on 485 examples. Resuming with 0 already completed.
[1/485] gsm8k_val_0: correct; valid_format=True; errors=[]
[2/485] gsm8k_val_1008: correct; valid_format=True; errors=[]
[3/485] gsm8k_val_1029: correct; valid_format=True; errors=[]
[4/485] gsm8k_val_1041: correct; valid_format=True; errors=[]
[5/485] gsm8k_val_1050: correct; valid_format=True; errors=[]
[6/485] gsm8k_val_1093: wrong; valid_format=True; errors=['wrong_answer']
[7/485] gsm8k_val_1178: correct; valid_format=True; errors=[]
[8/485] gsm8k_val_1187: wrong; valid_format=True; errors=['wrong_answer']
[9/485] gsm8k_val_1202: wrong; valid_format=True; errors=['wrong_answer']
[10/485] gsm8k_val_1214: correct; valid_format=True; errors=[]
[11/485] gsm8k_val_1247: correct; valid_format=True; errors=[]
[12/485] gsm8k_val_1285: correct; valid_format=True; errors=[]
[13/485] gsm8k_val_1287: wrong; valid_format=True; errors=['wrong_answer']
[14/485] gsm8k_val_1301: wr